# Metadata Transformation Pipeline (Prototype)

Seed notebook to prototype the OGD → Dublin Core transformation flow using the architecture guidance in `metadata_transformation_architecture.md`. It works against the sample OGD dataset metadata and catalog metadata provided in `data/` and leaves room to plug in richer mappers/validators later.


## Goals
- Load sample dataset and catalog metadata for quick iteration
- Sketch the ingest → normalize → map → derived → validate → export flow
- Keep helpers small and pure so they can be reused inside a production pipeline
- Show where data profiling/PII scans and catalog-level enrichment plug in


## Setup paths and imports
Update `REPO_ROOT` if you run this notebook from outside the repo root.


In [ ]:
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd

# Resolve repository root and sample metadata locations
REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent

DATASETS_METADATA_PATH = REPO_ROOT / '/home/aakash/NIC/Newfolder/nic-metadata-cleaning/data/sample_datasets_metadata/nic_sample_dataset.csv'
CATALOG_METADATA_PATH = REPO_ROOT / '/home/aakash/NIC/Newfolder/nic-metadata-cleaning/data/sample_catalog_metadata/nic_sample_catalog.csv'

print(f'Repo root: {REPO_ROOT}')
print(f'Dataset metadata path: {DATASETS_METADATA_PATH.exists()} -> {DATASETS_METADATA_PATH}')
print(f'Catalog metadata path: {CATALOG_METADATA_PATH.exists()} -> {CATALOG_METADATA_PATH}')


## Ingest: load raw OGD metadata
We keep everything as strings initially


In [ ]:
datasets_df = pd.read_csv(DATASETS_METADATA_PATH, dtype=str, keep_default_na=False, low_memory=False)
catalog_df = pd.read_csv(CATALOG_METADATA_PATH, dtype=str, keep_default_na=False, low_memory=False)

print(datasets_df.shape, 'datasets records')
print(catalog_df.shape, 'catalog records')


In [ ]:
# Peek at a few rows to understand available fields
pd.set_option('display.max_columns', 10)
preview_cols = [
    'title',
    'catalog_title',
    'published_date',
    'changed',
    'created',
    'sector',
    'field_resource_type',
    'datafile',
    'datafile_url',
    'file_format',
    'file_size',
    'frequency',
    'granularity',
    'node_alias',
    'domain',
]
datasets_df[preview_cols].head(3)


In [ ]:
catalog_df[[
    'title',
    'field_asset_jurisdiction:name',
    'field_ds_govt_type',
    'field_ministry_department:name',
    'field_sector:name',
    'keywords',
    'node_alias',
]].head(3)


## Flow outline (from architecture doc)
1. **Ingest** raw OGD dataset + catalog metadata.
2. **Normalize** strings/dates, clean NA values, and standardize lists.
3. **Map** OGD fields → Dublin Core targets (direct, composite, controlled vocab).
4. **Compute derived** fields for frequency, granularity, and access rights.
5. **Profile** dataset content (PII, schema, stats) — stubbed here.
6. **Validate** required fields and controlled vocab values.
7. **Export** JSON/CSV for downstream consumers.


## Normalization & mapping helpers
Simplified versions of the helpers described in the architecture spec. All helpers return `None` when input is empty/invalid so the mapper can decide on defaults later.


In [ ]:
NA_VALUES = {'', 'na', 'n/a', 'null', 'none', 'nan', '(empty)', 'NA', 'N/A', 'None'}
DATE_FORMATS = ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%m/%d/%Y', '%Y/%m/%d']
FREQUENCY_URI = {
    'daily': 'http://purl.org/cld/freq/continuous',
    'weekly': 'http://purl.org/cld/freq/weekly',
    'fortnightly': 'http://purl.org/cld/freq/biweekly',
    'monthly': 'http://purl.org/dc/terms/Monthly',
    'quarterly': 'http://purl.org/dc/terms/Quarterly',
    'half yearly': 'http://purl.org/dc/terms/Frequency/Semiannual',
    'annual': 'http://purl.org/dc/terms/Annual',
    'bi-annual': 'http://purl.org/cld/freq/biennial',
    'one-time': 'http://purl.org/cld/freq/irregular',
    'others': 'http://purl.org/cld/freq/irregular',
}
GRANULARITY_DURATION = {
    'hourly': 'PT1H',
    'daily': 'P1D',
    'weekly': 'P7D',
    'fortnightly': 'P14D',
    'monthly': 'P1M',
    'quarterly': 'P3M',
    'half yearly': 'P6M',
    'annual': 'P1Y',
    'bi-annual': 'P2Y',
    'one-time': 'NA',
    'others': 'NA',
}
ACCESS_RIGHTS = {
    '1': 'Open',  # static dataset
    '2': 'Open',  # periodic collection
    '4': 'Restricted',  # file download with possible auth
    '5': 'Registered',  # API service / registered access
}

def normalize_str(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip()
    if text == '':
        return None
    if text.lower() in NA_VALUES:
        return None
    return ' '.join(text.split())

def split_multi(value: Optional[str], sep: str = ';') -> List[str]:
    text = normalize_str(value)
    if not text:
        return []
    return [part.strip() for part in text.split(sep) if normalize_str(part)]

def parse_date_str(value: Optional[str]) -> Optional[str]:
    text = normalize_str(value)
    if not text:
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt).date().isoformat()
        except ValueError:
            continue
    return None

def map_frequency_uri(value: Optional[str]) -> Optional[str]:
    text = normalize_str(value)
    if not text:
        return None
    return FREQUENCY_URI.get(text.lower())

def map_granularity_duration(value: Optional[str]) -> Optional[str]:
    text = normalize_str(value)
    if not text:
        return None
    return GRANULARITY_DURATION.get(text.lower())

def ensure_https(url: Optional[str]) -> Optional[str]:
    text = normalize_str(url)
    if not text:
        return None
    if text.startswith('http://'):
        return 'https://' + text[len('http://'):]
    return text

def build_landing_page(domain: Optional[str], node_alias: Optional[str]) -> Optional[str]:
    dom = normalize_str(domain) or 'data.gov.in'
    alias = normalize_str(node_alias) or ''
    if not alias:
        return None
    if alias.startswith('/'):
        alias = alias[1:]
    return f'https://{dom}/{alias}'

def format_extent_bytes(file_size: Optional[str]) -> Optional[str]:
    size_str = normalize_str(file_size)
    if not size_str:
        return None
    try:
        as_float = float(size_str)
    except ValueError:
        return None
    as_int = int(as_float)
    return f'{as_int} bytes'


## Catalog lookup (optional enrichment)
Matches dataset rows to catalog metadata by title (case-insensitive). Adjust the key if you have a better join, e.g., by catalog UUID.


In [ ]:
catalog_index = {
    (row.get('title') or '').strip().lower(): row
    for row in catalog_df.to_dict(orient='records')
}

def find_catalog_record(dataset_row: Dict[str, Any]) -> Dict[str, Any]:
    title_key = (dataset_row.get('catalog_title') or '').strip().lower()
    return catalog_index.get(title_key, {})


## Map a single OGD record → Dublin Core-ish dict
This is intentionally simple and mirrors the architecture modules:
- **Normalize** inputs
- **Map** to DC fields (with catalog fallback where sensible)
- **Compute** derived values for frequency, granularity, and access rights


In [ ]:
def transform_record(row: Dict[str, Any]) -> Dict[str, Any]:
    catalog_row = find_catalog_record(row)
    dc: Dict[str, Any] = {}

    # Titles & descriptions
    dc['dc:title'] = normalize_str(row.get('title'))
    dc['dcterms:alternative'] = normalize_str(catalog_row.get('title')) if catalog_row else None
    dc['dc:description'] = normalize_str(row.get('note')) or normalize_str(catalog_row.get('body:value'))

    # Subjects
    dc['dc:subject.keyword'] = split_multi(row.get('sector')) or split_multi(catalog_row.get('keywords'))
    dc['dc:subject.sector'] = split_multi(row.get('sector'))
    dc['dc:subject.sector_resource'] = split_multi(row.get('sector_resource'))

    # Dates
    dc['dcterms:issued'] = parse_date_str(row.get('published_date'))
    dc['dcterms:modified'] = parse_date_str(row.get('changed'))
    dc['dcterms:created'] = parse_date_str(row.get('created'))

    # Identifier
    dc['dc:identifier.landing_page'] = build_landing_page(row.get('domain'), row.get('node_alias'))
    dc['dc:identifier.api_url'] = ensure_https(row.get('datafile_url'))

    # Relations & format
    dc['dc:relation.download_url'] = ensure_https(row.get('datafile'))
    dc['dc:relation.catalog_title'] = normalize_str(row.get('catalog_title'))
    dc['dc:format'] = normalize_str(row.get('file_format'))
    dc['dcterms:extent'] = format_extent_bytes(row.get('file_size'))

    # Frequency / coverage / access
    dc['dcterms:accrualPeriodicity'] = map_frequency_uri(row.get('frequency'))
    dc['dcterms:coverage'] = map_granularity_duration(row.get('granularity'))
    dc['dcterms:accessRights'] = ACCESS_RIGHTS.get((row.get('field_resource_type') or '').strip(), 'Open')

    # Publisher hierarchy
    publishers = split_multi(row.get('ministry_department')) + split_multi(row.get('state_department'))
    dc['dc:publisher'] = publishers if publishers else split_multi(catalog_row.get('field_ministry_department:name'))

    # License / rights placeholders (not present in sample metadata)
    dc['dcterms:license'] = None
    dc['dc:rights'] = None

    return dc


## Run the mapper on a sample slice
We only transform a small subset to keep the notebook lightweight.


In [ ]:
sample_records = datasets_df.head(5)
transformed_records: List[Dict[str, Any]] = []
for _, record in sample_records.iterrows():
    transformed_records.append(transform_record(record.to_dict()))

pd.DataFrame(transformed_records)


## Validation stub
Minimal checks for required fields (title, issued/modified dates, landing page). Extend with controlled vocab validation using the YAML configs referenced in the architecture doc.


In [ ]:
REQUIRED_FIELDS = ['dc:title', 'dcterms:issued', 'dcterms:modified', 'dc:identifier.landing_page']

def validate_record(dc_record: Dict[str, Any]) -> Dict[str, Any]:
    issues = []
    for field in REQUIRED_FIELDS:
        if not dc_record.get(field):
            issues.append({'field': field, 'severity': 'error', 'message': 'missing'})

    return {'is_valid': len([i for i in issues if i['severity'] == 'error']) == 0, 'issues': issues}

validation_results = [validate_record(dc) for dc in transformed_records]
validation_results


## Derived fields & profiling placeholders
- Temporal coverage: use the dataset file to compute `dcterms:temporal` (see `TemporalCoverageCalculator` in the architecture doc). This notebook does not fetch remote files.
- PII/schema profiling: run a profiler over the actual dataset file and attach warnings to the record before export.
- Endpoint description: generate when `datafile_url` is present.


## Export sample (in-memory)
This is kept in-memory for now; uncomment the write block to persist.


In [ ]:
# Convert to DataFrame for quick exports
transformed_df = pd.DataFrame(transformed_records)
transformed_df.head()

# Uncomment to write a small JSON preview next to the notebook
# output_path = Path('notebooks/results/metadata_transformation_sample.json')
# output_path.parent.mkdir(parents=True, exist_ok=True)
# output_path.write_text(transformed_df.to_json(orient='records', indent=2), encoding='utf-8')
# output_path


## Next steps
- Replace the inline helper dicts with YAML-driven configs (`config/mapping`, `config/validation`).
- Plug in dataset-level profilers for temporal coverage, schema, and PII detection.
- Add richer validation rules (controlled vocabularies, cross-field constraints).
- Wire this notebook logic into `metadata_transformation_process.py` or a CLI entrypoint for repeatable runs.
